In [18]:
import os
import ast
import json
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


train_data = read_json("../data/train_spider.json")
train_tables = read_json("../data/tables.json")
dev_data = read_json("../data/dev.json")

print(f"Training questions: {len(train_data):,}")
print(f"Training tables:    {len(train_tables):,}")
print(f"Dev questions:      {len(dev_data):,}")


Training questions: 7,000
Training tables:    166
Dev questions:      1,034


In [9]:
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(dev_data)

In [42]:
train_df.sample(2)

,db_id,query,query_toks,query_toks_no_value,question,question_toks,sql,input_text,target_text
6500,scientist_1,"SELECT count(*) , T1.name FROM scientists AS ...","[SELECT, count, (, *, ), ,, T1.name, FROM, sci...","[select, count, (, *, ), ,, t1, ., name, from,...","What are the names of the scientists, and how ...","[What, are, the, names, of, the, scientists, ,...","{'from': {'table_units': [['table_unit', 0], [...",translate question to SQL: database schema: sc...,"SELECT count(*) , T1.name FROM scientists AS ..."
2944,sakila_1,SELECT payment_date FROM payment WHERE amount ...,"[SELECT, payment_date, FROM, payment, WHERE, a...","[select, payment_date, from, payment, where, a...",Find all the payment dates for the payments wi...,"[Find, all, the, payment, dates, for, the, pay...","{'from': {'table_units': [['table_unit', 12]],...",translate question to SQL: database schema: ac...,SELECT payment_date FROM payment WHERE amount ...


In [12]:
train_table_df = pd.DataFrame(train_tables)
train_table_df.head(1)

,column_names,column_names_original,column_types,db_id,foreign_keys,primary_keys,table_names,table_names_original
0,"[[-1, *], [0, perpetrator id], [0, people id],...","[[-1, *], [0, Perpetrator_ID], [0, People_ID],...","[text, number, number, text, number, text, tex...",perpetrator,"[[2, 9]]","[1, 9]","[perpetrator, people]","[perpetrator, people]"


In [21]:
train_df = train_df.copy()
train_table = train_table_df.copy()

train_df.columns = train_df.columns.str.strip()
train_table.columns = train_table.columns.str.strip()

print(train_df[["db_id", "question", "query"]].head())

                   db_id                                           question  \
0  department_management  How many heads of the departments are older th...   
1  department_management  List the name, born state and age of the heads...   
2  department_management  List the creation year, name and budget of eac...   
3  department_management  What are the maximum and minimum budget of the...   
4  department_management  What is the average number of employees of the...   

                                               query  
0         SELECT count(*) FROM head WHERE age  >  56  
1  SELECT name ,  born_state ,  age FROM head ORD...  
2  SELECT creation ,  name ,  budget_in_billions ...  
3  SELECT max(budget_in_billions) ,  min(budget_i...  
4  SELECT avg(num_employees) FROM department WHER...  


### Dataframe refinement

In [23]:
train_df = train_df.dropna(
    subset=["db_id", "question", "query"]
).copy()

train_df["db_id"] = train_df["db_id"].astype(str).str.strip()
train_df["question"] = train_df["question"].astype(str).str.strip()
train_df["query"] = train_df["query"].astype(str).str.strip()

train_df = train_df[
    (train_df["question"] != "") &
    (train_df["query"] != "")
].copy()

train_df = train_df.reset_index(drop=True)

print("Training examples:", len(train_df))

Training examples: 7000


In [ ]:
#Convert schema fields into Python lists
def to_list(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, TypeError):
            try:
                return ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []

    return []

In [ ]:
#Create a schema for each database
required_schema_columns = [
    "db_id",
    "table_names",
    "column_names"
]

missing = [
    col for col in required_schema_columns
    if col not in train_table.columns
]

if missing:
    raise ValueError(
        f"Missing schema columns: {missing}. "
        "Check train_table.columns."
    )

schema_lookup = {}

for _, row in train_table.iterrows():
    db_id = str(row["db_id"]).strip()

    tables = to_list(row["table_names"])
    columns = to_list(row["column_names"])

    schema_lookup[db_id] = {
        "tables": tables,
        "columns": columns,
        "column_types": to_list(row.get("column_types", [])),
        "primary_keys": to_list(row.get("primary_keys", [])),
        "foreign_keys": to_list(row.get("foreign_keys", []))
    }

print("Databases with schemas:", len(schema_lookup))

Databases with schemas: 166


In [26]:
#Build readable database schemas


def format_schema(db_id):
    schema = schema_lookup.get(db_id)

    if schema is None:
        return ""

    tables = schema["tables"]
    columns = schema["columns"]

    lines = []

    for i, table_name in enumerate(tables):
        table_columns = []

        for col in columns:
            if (
                isinstance(col, (list, tuple)) and
                len(col) >= 2 and
                col[0] == i
            ):
                table_columns.append(str(col[1]))

        lines.append(
            f"{table_name} ("
            + ", ".join(table_columns)
            + ")"
        )

    return " | ".join(lines)

In [27]:
example_db = train_df.iloc[0]["db_id"]

print("Database:", example_db)
print("Schema:", format_schema(example_db))

Database: department_management
Schema: department (department id, name, creation, ranking, budget in billions, num employees) | head (head id, name, born state, age) | management (department id, head id, temporary acting)


In [28]:
#Build training inputs and SQL targets
def make_input(row):
    schema = format_schema(row["db_id"])

    return (
        "translate question to SQL: "
        + "database schema: "
        + schema
        + " question: "
        + row["question"]
    )

train_df["input_text"] = train_df.apply(make_input, axis=1)
train_df["target_text"] = train_df["query"]

print(train_df[["input_text", "target_text"]].head(3))

                                          input_text  \
0  translate question to SQL: database schema: de...   
1  translate question to SQL: database schema: de...   
2  translate question to SQL: database schema: de...   

                                         target_text  
0         SELECT count(*) FROM head WHERE age  >  56  
1  SELECT name ,  born_state ,  age FROM head ORD...  
2  SELECT creation ,  name ,  budget_in_billions ...  


In [43]:
#model import
model_path = "finassist_t5_wikisql"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Model loaded:", model.__class__.__name__)
print("Device:", device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded: T5ForConditionalGeneration
Device: cuda


In [30]:
#Split training and validation data
from sklearn.model_selection import train_test_split

train_part, val_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42
)

print("Training examples:", len(train_part))
print("Validation examples:", len(val_part))

Training examples: 6300
Validation examples: 700


In [31]:
#Tokenize inputs and SQL targets
def preprocess(batch):
    inputs = tokenizer(
        batch["input_text"],
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        text_target=batch["target_text"],
        max_length=256,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs


In [32]:
#Convert the Pandas DataFrames into Hugging Face datasets and tokenize them:
train_dataset = Dataset.from_pandas(
    train_part[["input_text", "target_text"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_part[["input_text", "target_text"]],
    preserve_index=False
)

train_dataset = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    preprocess,
    batched=True,
    remove_columns=val_dataset.column_names
)

print(train_dataset)
print(val_dataset)




Map:   0%|          | 0/6300 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 6300
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 700
})


In [33]:
training_args = Seq2SeqTrainingArguments(
    output_dir="finassist_t5_spider",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42
)

In [34]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

In [35]:
train_result = trainer.train()

print(train_result)

Epoch,Training Loss,Validation Loss
1,7.651323,1.372877
2,5.671087,1.064046
3,5.427870,1.001221


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2364, training_loss=7.4198646351770705, metrics={'train_runtime': 1720.7067, 'train_samples_per_second': 10.984, 'train_steps_per_second': 1.374, 'total_flos': 1045466249232384.0, 'train_loss': 7.4198646351770705, 'epoch': 3.0})


## Save the model

In [41]:
trainer.save_model("finassist_t5_spider/final_model")
tokenizer.save_pretrained("finassist_t5_spider/final_model")

# print("Saved to:", FINAL_MODEL_DIR.resolve())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('finassist_t5_spider/final_model\\tokenizer_config.json',
 'finassist_t5_spider/final_model\\tokenizer.json')

In [40]:
import torch

model.eval()
device = next(model.parameters()).device

# 1. Define the question
question = "What are the names of singers who have won more than 2 awards?"

# 2. Define the schema
schema = """
Database: singer

Table: singer
Columns:
singer_id, name, birth_year, net_worth, citizenship

Table: awards
Columns:
award_id, singer_id, award_name, award_year
"""

# 3. Prepare input in the same format used during training
test_input = (
    "translate question to SQL: "
    + "database schema: "
    + schema
    + " question: "
    + question
)

# 4. Tokenize input
inputs = tokenizer(
    test_input,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

# 5. Generate SQL using your trained model
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4
    )

# 6. Decode prediction
predicted_sql = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# 7. Display the result
print("Question:")
print(question)

print("\nSchema:")
print(schema)

print("\nPredicted SQL:")
print(predicted_sql)

Question:
What are the names of singers who have won more than 2 awards?

Schema:

Database: singer

Table: singer
Columns:
singer_id, name, birth_year, net_worth, citizenship

Table: awards
Columns:
award_id, singer_id, award_name, award_year


Predicted SQL:
SELECT name FROM singer AS T1 JOIN awards AS T2 ON T1.song_id = T2.song_id WHERE T1.song_id = T2.song_id WHERE T1.song_id = T2.song_id WHERE T1.song_id = T2.song_id WHERE T1.song_id = T2.song_id WHERE T2.song_id


In [38]:
model.eval()

test_question = train_part.iloc[0]["question"]
test_db_id = train_part.iloc[0]["db_id"]

test_input = (
    "translate question to SQL: "
    + "database schema: "
    + format_schema(test_db_id)
    + " question: "
    + test_question
)

inputs = tokenizer(
    test_input,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(next(model.parameters()).device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128
    )

predicted_sql = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", test_question)
print("Database:", test_db_id)
print("Actual SQL:", train_part.iloc[0]["query"])
print("Predicted SQL:", predicted_sql)

Question: What are the details and star ratings of the three hotels with the lowest price ranges?
Database: cre_Theme_park
Actual SQL: SELECT other_hotel_details ,  star_rating_code FROM HOTELS ORDER BY price_range ASC LIMIT 3
Predicted SQL: SELECT star_rated_code FROM hotels WHERE price_range = 3
